## Configure paths and initialize a Keypoint-MoSeq project from an existing DeepLabCut project.

In [ ]:
import keypoint_moseq as kpms
import matplotlib.pyplot as plt
from pathlib import Path

WORKDIR = Path.cwd().parent
dlc_project_dir = WORKDIR.parent / "dlc-pose-estimation" / "ElevatedMazeFood-Atanu-2026-04-04"
project_dir = WORKDIR / "results" / "ElevatedMazeFood"
dlc_config = dlc_project_dir / "config.yaml"

print(f"dlc_project_dir: {dlc_project_dir}")
print(f"dlc_config exists: {dlc_config.exists()} -> {dlc_config}")
print(f"kpms project_dir: {project_dir}")

In [ ]:
# Create the KPMS project once, then define a relaxed config loader.
if not (project_dir / "config.yml").exists():
    kpms.setup_project(project_dir, deeplabcut_config=str(dlc_config), overwrite=True)

# Use a relaxed load first in case anatomy fields still need editing.
config = lambda: kpms.load_config(project_dir, check_if_valid=False, build_indexes=False)

## Tutorial reference (optional)
The next cell is a commented tutorial-style variant and is not required for this workflow.

In [ ]:
# Optional tutorial snippet kept for reference only (do not run in this workflow).
# from pathlib import Path
# import keypoint_moseq as kpms

# WORKDIR = Path.cwd().parent
# project_dir = "demo_project"
# dlc_project_dir = WORKDIR.parent / "dlc-pose-estimation" / "ElevatedMazeFood-Atanu-2026-04-04"
# config = lambda: kpms.load_config(project_dir)
# dlc_config = dlc_project_dir / "config.yaml"
# kpms.setup_project(project_dir, deeplabcut_config=dlc_config)

In [ ]:
# Set core project config values for this dataset.
kpms.update_config(
    project_dir,
    video_dir=str(dlc_project_dir / "videos"),
    anterior_bodyparts=["Head"],
    posterior_bodyparts=["Tailbase"],
    fps=15,
)

# After anatomy values are valid, switch back to strict config loading.
config = lambda: kpms.load_config(project_dir)

## Load Data
Load raw keypoints and keep untouched backups so selected keypoints can be restored later if needed.

In [ ]:
# load data (e.g. from DeepLabCut)
keypoint_data_path = str(dlc_project_dir / "raw_pose_data")  # can be a file, a directory, or a list of files
coordinates, confidences, bodyparts = kpms.load_keypoints(keypoint_data_path, "deeplabcut")

# Preserve originals for optional per-keypoint restoration after outlier interpolation.
orig_coordinates = {k: v.copy() for k, v in coordinates.items()}
orig_confidences = {k: v.copy() for k, v in confidences.items()}

## Remove outlier keypoints
Run medoid-distance outlier interpolation, then restore original Midback values to avoid over-correction of a central keypoint.

In [ ]:
# Configure outlier sensitivity and run outlier interpolation on temporary copies.
kpms.update_config(project_dir, outlier_scale_factor=4.0)

# outlier_removal mutates inputs in place, so work on temporary copies.
temp_coordinates = {k: v.copy() for k, v in coordinates.items()}
temp_confidences = {k: v.copy() for k, v in confidences.items()}

temp_coordinates, temp_confidences = kpms.outlier_removal(
    temp_coordinates,
    temp_confidences,
    project_dir,
    overwrite=True,
    **config()
)

# Restore Midback from original DLC output to avoid central-keypoint over-correction.
mid_idx = bodyparts.index("Midback")

for rec in temp_coordinates:
    temp_coordinates[rec][:, mid_idx, :] = orig_coordinates[rec][:, mid_idx, :]
    temp_confidences[rec][:, mid_idx] = orig_confidences[rec][:, mid_idx]

coordinates, confidences = temp_coordinates, temp_confidences

In [ ]:
# Save cleaned keypoints so later steps can resume without rerunning preprocessing.
import pickle

cleaned_snapshot = project_dir / "cleaned_keypoints.pkl"
with open(cleaned_snapshot, "wb") as f:
    pickle.dump(
        {
            "coordinates": coordinates,
            "confidences": confidences,
            "bodyparts": bodyparts,
        },
        f,
    )

print(f"Saved cleaned keypoints snapshot: {cleaned_snapshot}")

## Format data for modeling
Optionally reload the cleaned snapshot, then convert keypoints into model-ready arrays.

In [ ]:
# Optional resume path: use this after kernel restart to skip preprocessing cells.
import pickle

cleaned_snapshot = project_dir / "cleaned_keypoints.pkl"
if cleaned_snapshot.exists():
    with open(cleaned_snapshot, "rb") as f:
        snap = pickle.load(f)
    coordinates = snap["coordinates"]
    confidences = snap["confidences"]
    bodyparts = snap["bodyparts"]
    print(f"Loaded cleaned keypoints snapshot: {cleaned_snapshot}")
else:
    print(f"Snapshot not found: {cleaned_snapshot}")

In [ ]:
# Build batched arrays and metadata used by downstream KPMS model fitting.
data, metadata = kpms.format_data(coordinates, confidences, **config())

In [ ]:
import pickle

formatted_snapshot = project_dir / "formatted_data.pkl"
with open(formatted_snapshot, "wb") as f:
    pickle.dump({"data": data, "metadata": metadata}, f)

print(f"Saved formatted snapshot: {formatted_snapshot}")

## Update sigmasq_loc

In [ ]:
estimated_sigmasq_loc = kpms.estimate_sigmasq_loc(
    data["Y"], data["mask"], filter_size=config()["fps"]
)
print("estimated sigmasq_loc:", estimated_sigmasq_loc)

kpms.update_config(project_dir, sigmasq_loc=float(estimated_sigmasq_loc))
print("updated sigmasq_loc:", config()["cen_hypparams"]["sigmasq_loc"])

## Calibration
Estimate confidence-to-error mapping parameters using manual annotation support.

In [ ]:
%matplotlib widget
kpms.noise_calibration(project_dir, coordinates, confidences, **config())

## Fit PCA

In [ ]:
plt.close("all")
%matplotlib inline
pca = kpms.fit_pca(**data, **config())
kpms.save_pca(pca, project_dir)

kpms.print_dims_to_explain_variance(pca, 0.9)
kpms.plot_scree(pca, project_dir=project_dir)
kpms.plot_pcs(pca, project_dir=project_dir, **config())

# use the following to load an already fit model
# pca = kpms.load_pca(project_dir)

## Set latent dimensionality before model initialization

In [ ]:
kpms.update_config(project_dir, latent_dim=4)